# Analyze a whole collection of h5ad files as one dataset

Single-cell studies usually arrive as **one `.h5ad` file per sample** (per donor,
per run, per cell type). To analyze them together you normally load every file
and concatenate it in memory — which copies the data and, for a large study, may
simply not fit.

This notebook shows a different path: point at the folder once, and from then on
work with the **entire collection as a single `AnnData`** — without copying the
matrices or loading them all into memory. Cells are read from the original files
only when you actually touch them.

*(The virtualization machinery lives in the `virtual_h5ad` package; you
don't need to know how it works to use it.)*

## A collection to work with

We stand in for a multi-sample study by splitting the classic pbmc3k dataset into
three labelled "samples". In your work these would be your real per-sample files.

In [4]:
import glob, os
import anndata as ad
import numpy as np

os.makedirs('data/samples', exist_ok=True)
if not os.path.exists('data/pbmc3k_modern.h5ad'):
    import urllib.request
    urllib.request.urlretrieve('https://falexwolf.de/data/pbmc3k_raw.h5ad', 'data/pbmc3k_raw.h5ad')
    ad.read_h5ad('data/pbmc3k_raw.h5ad').write_h5ad('data/pbmc3k_modern.h5ad', compression='gzip')

if not glob.glob('data/samples/*.h5ad'):
    a = ad.read_h5ad('data/pbmc3k_modern.h5ad')
    edges = np.linspace(0, a.n_obs, 4).astype(int)
    for i in range(3):
        s = a[edges[i]:edges[i + 1]].copy()
        s.obs['sample'] = f'sample_{i}'
        s.write_h5ad(f'data/samples/sample_{i}.h5ad', compression='gzip')

samples = sorted(glob.glob('data/samples/*.h5ad'))
total_mb = sum(os.path.getsize(p) for p in samples) / 1e6
print(f'{len(samples)} sample files, {total_mb:.1f} MB total:')
for p in samples:
    print('  ', os.path.basename(p))

3 sample files, 12.0 MB total:
   sample_0.h5ad
   sample_1.h5ad
   sample_2.h5ad


## Step 1 — register the collection (no copying)

One call points at the folder and writes a small repository that **references** the
original files. Notice how little disk it uses — the gene-expression matrices are
not duplicated.

In [5]:
from virtual_h5ad import virtualize_matrix

virtualize_matrix('data/samples', 'repo/samples')

repo_kb = sum(os.path.getsize(os.path.join(r, f))
              for r, _, fs in os.walk('repo/samples') for f in fs) / 1e3
print(f'{total_mb:.1f} MB of sample files  ->  {repo_kb:.0f} KB repository '
      f'({total_mb * 1000 / repo_kb:.0f}x smaller)')

12.0 MB of sample files  ->  350 KB repository (34x smaller)


/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)


## Step 2 — open the collection as one AnnData

`open_anndata` gives you a normal `AnnData` spanning **all** the samples. The cell
metadata (`obs`) from every file is stitched together, so you can immediately ask
questions across the whole study.

In [6]:
from virtual_h5ad import open_anndata

adata = open_anndata('repo/samples')
adata

AnnData object with n_obs × n_vars = 2700 × 32738
    obs: 'sample'
    var: 'gene_ids'

In [7]:
# It behaves like any AnnData — here, how many cells came from each sample:
adata.obs['sample'].value_counts()

sample
sample_0    900
sample_1    900
sample_2    900
Name: count, dtype: int64

## Step 3 — pull any set of cells, across samples, on demand

Because the collection is a single dataset, selecting cells that span several files
is just an ordinary selection — and only the cells you ask for are read from disk.
Below we take the first 50 cells of *each* sample and get one expression matrix
back, without ever loading the full collection.

In [8]:
import numpy as np

# row positions of the first 50 cells of every sample
picks = np.flatnonzero(adata.obs.groupby('sample', observed=True).cumcount() < 50)

# just an ordinary AnnData slice: only those rows are read from the source files
subset = adata[picks].to_memory()
print(f'selected {subset.n_obs} cells from {adata.obs["sample"].nunique()} samples '
      f'-> matrix {subset.shape}, {subset.X.nnz} nonzero entries')
print('mean expression in the selection:', round(float(subset.X.mean()), 4))

selected 150 cells from 3 samples -> matrix (150, 32738), 124988 nonzero entries
mean expression in the selection: 0.0722


## When to reach for this

- You have many per-sample `.h5ad` files that share the same genes and want to
  treat them as one dataset.
- The combined data is large enough that copying or loading it all is painful.
- You only need parts of it at a time (specific cells, samples, or cell types).

The same `virtualize_matrix(folder, repo)` call scales to real studies of many
large files (it's been run on the Allen Institute *Human Immune Health Atlas*).
Curious how it avoids copying? See `src/virtual_h5ad/_concat.py` and
`FINDINGS.md`.